# Track Reconstruction Performance Analysis

This notebook analyzes the performance of two track reconstruction algorithms (Classical Kalman Filter and Global Kalman Filter) applied to particle tracking data. 

The analysis focuses on comparing momentum reconstruction quality across different particle types (muons, pions, protons) by evaluating:

- **Resolution and bias** as a function of:
    - Number of tracking points
    - Track length
    - Lever arm (distance between first and last measurement)
    - True particle momentum

Results are visualized through profile plots showing both the resolution (RMS spread) and systematic bias in momentum reconstruction, with output saved as `.eps` and `.png` files in the `Res/` directory.

NOTE: You will need a decent amount of statistics to see meaningful results (of the order of about 10000 tracks per paricle type at least).

## Loading libraries

In [ ]:
import os
import ROOT
from Plot_func import Draw3HistosRes
from Plot_func import SetGlobalStyle
from ROOT import gROOT, gSystem
import os
gSystem.Load("../aliKalman/AliExternalTrackParam.so")
gROOT.LoadMacro("../MC/fastSimulation.cxx+")
gROOT.LoadMacro("../MC/fastSimulationTest.C+")

# Reading the files

In [ ]:
folder = "../data/"
Ideal = False
ptype = "pgun"
pname = ""
if (ptype == "2212"): pname = "protons"
if (ptype == "13"): pname = "muons" 
if (ptype == "211"): pname = "pions" 
if (ptype == "pgun"): pname = "muons" 

paramletter = ["(a)","(d)"]
if (ptype == "13"): paramletter = ["(a)","(d)"]
if (ptype == "211"): paramletter = ["(b)","(e)"]
if (ptype == "2212"): paramletter = ["(c)","(f)"]


foldercheck="Res/"
os.makedirs(foldercheck, exist_ok=True)

inputData = folder+"fastParticle.list"
print(inputData)
tree  = ROOT.AliXRDPROOFtoolkit.MakeChainRandom(inputData,"fastPart",chr(0),10000)

## NPoints

In [ ]:
SetGlobalStyle()

Y = "(part.fParamMC[0].GetP()-part.fParamIn[0].GetP())/part.fParamMC[0].GetP()"
X = "part.fParamMC@.size()"
Xname = "#it{N}"
xrange = "4,50,450"
yrange = "60,-0.6,0.6"
yrangeuser = [0,0.3]
extracond = ["part.fParamIn[0].fP[4]!=0 && part.fParamIn@.size()>50 && abs(pdgCode)==2212",
             "part.fParamIn[0].fP[4]!=0 && part.fParamIn@.size()>50 && abs(pdgCode)==13",
             "part.fParamIn[0].fP[4]!=0 && part.fParamIn@.size()>50 && abs(pdgCode)==211"]

cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.4, 0.74, 0.94, 0.9)
lgn = ["CKF "+pname,"GKF "+pname]


Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)

lg.Draw()
cc.Draw()
cc.Print("Res/Resp_vs_NPoints.eps")
cc.Print("Res/Resp_vs_NPoints.png")

In [ ]:
yrangeuser = [-0.05,0.05]
cc2=ROOT.TCanvas("cc2","",600,600)
lg2 = ROOT.TLegend(0.47, 0.72, 0.94, 0.9)

Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc2,lg2,yrangeuser,1)

lg2.Draw()
cc2.Draw()
cc2.Print("Res/Biasp_vs_NPoints.eps")
cc2.Print("Res/Biasp_vs_NPoints.png")

## Length

In [ ]:
X = "Length"
Xname = "#it{l} (cm)"
xrange = "6,0,600"

yrange = "30,-0.6,0.6"
yrangeuser = [0,0.18]

cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.4, 0.74, 0.94, 0.9)

Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)
lg.Draw()
cc.Draw()
cc.Print("Res/Resp_vs_Length.eps")
cc.Print("Res/Resp_vs_Length.png")

In [ ]:
yrangeuser = [-0.05,0.05]
if(ptype=="211") : yrangeuser = [-0.05,0.1]
if(ptype=="2212") : yrangeuser = [-0.1,0.2]
cc2=ROOT.TCanvas("cc2","",600,600)
lg2 = ROOT.TLegend(0.47, 0.72, 0.94, 0.9)
Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc2,lg2,yrangeuser,1)
lg2.Draw()
cc2.Draw()
cc2.Print("Res/Biasp_vs_Length.eps")
cc2.Print("Res/Biasp_vs_Length.png")

## LArm

In [ ]:
tree.SetAlias("gxMCSt","(part.fParamMC[0].fX*cos(part.fParamMC[0].fAlpha)-part.fParamMC[0].fP[0]*sin(part.fParamMC[0].fAlpha))")
tree.SetAlias("gyMCSt","(part.fParamMC[0].fX*sin(part.fParamMC[0].fAlpha)+part.fParamMC[0].fP[0]*cos(part.fParamMC[0].fAlpha))")
tree.SetAlias("gxMCEnd","(part.fParamMC[part.fParamMC@.size()-1].fX*cos(part.fParamMC[part.fParamMC@.size()-1].fAlpha)-part.fParamMC[part.fParamMC@.size()-1].fP[0]*sin(part.fParamMC[part.fParamMC@.size()-1].fAlpha))")
tree.SetAlias("gyMCEnd","(part.fParamMC[part.fParamMC@.size()-1].fX*sin(part.fParamMC[part.fParamMC@.size()-1].fAlpha)+part.fParamMC[part.fParamMC@.size()-1].fP[0]*cos(part.fParamMC[part.fParamMC@.size()-1].fAlpha))")
tree.SetAlias("LArmMC","sqrt((gxMCSt-gxMCEnd)*(gxMCSt-gxMCEnd)+(gyMCSt-gyMCEnd)*(gyMCSt-gyMCEnd))")

X = "LArmMC"
Xname = "#it{L}_{Arm} (cm)"
xrange = "5,0,500"
yrange = "30,-0.6,0.6"
yrangeuser = [0,0.15]


cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.4, 0.74, 0.94, 0.9)
Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)
lg.Draw()
cc.Draw()
cc.Print("Res/Resp_vs_LArm.eps")
cc.Print("Res/Resp_vs_LArm.png")

In [ ]:
yrangeuser = [-0.05,0.05]
if(ptype=="211"): yrangeuser = [-0.05,0.1]
if(ptype=="2212"): yrangeuser = [-0.1,0.2]
cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.4, 0.74, 0.94, 0.9)

Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)
lg.Draw()
cc.Draw()
cc.Print("Res/Biasp_vs_LArm.eps")
cc.Print("Res/Biasp_vs_LArm.png")

## Momentum p

In [ ]:
X = "part.fParamMC[0].GetP()"
Xname = "#it{p}_{true} (GeV/#it{c})"
xrange = "11,0,5.5"
if(ptype=="2212" or ptype=="211"): xrange = "6,0,3"
yrange = "30,-0.6,0.6"
yrangeuser = [0,0.14]
if(ptype=="2212"): yrangeuser = [0,0.2]
if(ptype=="pgun") : xrange = "10,0.5,3"
if(ptype=="pgun") : yrange = "30,-0.3,0.3"
if(ptype=="pgun") : yrangeuser = [0,0.06]

if(ptype=="pgun"): paramletter[0] = "(a)"
if(ptype=="pgun"): paramletter[1] = "(c)"

cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.33, 0.74, 0.94, 0.9)

Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)
lg.Draw()
cc.Draw()
cc.Print("Res/Resp_vs_p.eps")
cc.Print("Res/Resp_vs_p.png")

In [ ]:
yrangeuser = [-0.05,0.05]
if(ptype=="2212"): yrangeuser = [-0.3,0.3]
if(ptype=="211"): yrangeuser = [-0.1,0.1]
cc=ROOT.TCanvas("cc","",600,600)
lg = ROOT.TLegend(0.2, 0.2, 0.74, 0.36)

Draw3HistosRes(tree,Y,X,xrange,yrange,Xname,extracond,cc,lg,yrangeuser)
lg.Draw()
cc.Draw()
cc.Print("Res/Biasp_vs_p.eps")
cc.Print("Res/Biasp_vs_p.png")